In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Install all packages from
!pip install -r ../requirements.txt

In [2]:
import json, os, csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [10]:
df = pd.read_csv("../../stats/download_stats1.csv")
df.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19416\3873842107.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../stats/download_stats1.csv")


,recid,file,bytes,start,end,duration_s,rate_Bps,success,error,expected_size,expected_checksum,computed_checksum_numeric,checksum_ok,attempts
0,80502,downloads/80502/99_175_prog_239.pdf,174717.0,1.764765e+09,1.764765e+09,0.962363,181549.996363,True,NaN,174717.0,adler32:9a26470d,2.586200e+09,True,NaN
1,80501,downloads/80501/cargo600.pdf,404470.0,1.764765e+09,1.764765e+09,1.017225,397620.874861,True,NaN,404470.0,adler32:18d0bebc,4.163335e+08,True,NaN
2,80502,downloads/80502/skelana.pdf,264402.0,1.764765e+09,1.764765e+09,0.935816,282536.223607,True,NaN,264402.0,adler32:4bac9f7a,1.269604e+09,True,NaN
3,80504,downloads/80504/dstana.pdf,489206.0,1.764765e+09,1.764765e+09,1.045380,467969.383739,True,NaN,489206.0,adler32:7304ac95,1.929686e+09,True,NaN
4,80503,downloads/80503/delgra2.0.pdf,255295.0,1.764765e+09,1.764765e+09,0.862576,295968.120697,True,NaN,255295.0,adler32:6eaeebd0,1.856957e+09,True,NaN


In [11]:
# print all unique files minus files with success == True

unique_files = set(df["file"].unique())
done_files = set(df[df["success"] == True]["file"].unique())
problematic_files = unique_files - done_files
print(f"Unique files: {len(unique_files)}")
print(f"Done files: {len(done_files)}")
print(f"Problematic files (not successful): {len(problematic_files)}")

Unique files: 414743
Done files: 414719
Problematic files (not successful): 24


In [12]:
print("Problematic files:")
for f in problematic_files:
    print(f)

Problematic files:
downloads/92941/EY1839.81.al
downloads/92941/EY1843.16.al
downloads/92941/EY1788.61.al
downloads/92941/EY1777.26.al
downloads/92941/EY1823.55.al
downloads/92941/EY1824.98.al
downloads/92941/EY1831.42.al
downloads/92941/EY1825.1.al
downloads/92941/EY1832.89.al
downloads/92941/EY1777.21.al
downloads/92941/EY1775.39.al
downloads/92941/EY1827.5.al
downloads/92941/EY1796.27.al
downloads/92941/EY1827.43.al
downloads/92941/EY1843.9.al
downloads/83967/run17640.xsdst
downloads/92941/EY1777.17.al
downloads/92941/EY1823.48.al
downloads/83967/run17609.xsdst
downloads/92941/EY1804.57.al
downloads/92941/EY1804.54.al
downloads/92941/EY1825.6.al
downloads/92941/EY1823.38.al
downloads/92941/EY1825.7.al


In [ ]:
# Count unique recids in lists/datasets
folder = "../lists"
res = set()
for filename in os.listdir(folder):
    if filename.endswith(".json") and filename.startswith('delphi-datasets'):
        with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                recids = set()
                def extract_recids(obj):
                    if isinstance(obj, dict):
                        for key, value in obj.items():
                            if key in ('recid', 'id'):
                                try:
                                    recids.add(int(value))
                                except (TypeError, ValueError):
                                    pass
                            extract_recids(value)
                    elif isinstance(obj, list):
                        for item in obj:
                            extract_recids(item)
                extract_recids(data)
                res.update(recids)
            except json.JSONDecodeError:
                pass
print(f"Unique recids in lists/datasets: {len(res)}")

In [ ]:
# find recid in res
recid = 93080
if recid in res:
    print(f"Recid {recid} found in lists/datasets.")
else:
    print(f"Recid {recid} NOT found in lists/datasets.")

In [29]:
with open('delphi_records_master.json', 'r') as cache_file:
    master_cache = json.load(cache_file)

In [24]:
# find recid x in master_cache
recid = 83967
recid_str = str(recid)
if recid_str in master_cache:
    print(f"Recid {recid} found in master_cache.")
else:
    print(f"Recid {recid} NOT found in master_cache.")


Recid 83967 found in master_cache.


In [30]:
not_done = []
for recid, data in master_cache.items():
    if not data.get("done", False):
        not_done.append(recid)

print(f"Recids in master_cache that are not downloaded: {len(not_done)}")

Recids in master_cache that are not downloaded: 67


In [31]:
not_checked = []
for recid, data in master_cache.items():
    if not data.get("checked", False):
        not_checked.append(recid)

print(f"Recids in master_cache that are not checked: {len(not_checked)}")

Recids in master_cache that are not checked: 36


In [32]:
missing = []
for recid, data in master_cache.items():
    for i in data.get("files", []):
        if not i.get("downloaded"):
            missing.append((recid, i.get("remote")))

print(f"Files in master_cache that are not downloaded: {len(missing)}")

Files in master_cache that are not downloaded: 120


In [23]:
print("Missing files:")
for recid, path in missing:
    print(f"Recid {recid}: {path}")

Missing files:
Recid 83114: root://eospublic.cern.ch//eos/opendata/delphi/collision-data/R07132/R07132.62.al
Recid 83114: root://eospublic.cern.ch//eos/opendata/delphi/collision-data/R07132/R07132.109.al
Recid 83114: root://eospublic.cern.ch//eos/opendata/delphi/collision-data/R07132/R07132.110.al
Recid 83114: root://eospublic.cern.ch//eos/opendata/delphi/collision-data/R07132/R07132.114.al
Recid 83583: root://eospublic.cern.ch//eos/opendata/delphi/simulated-data/karlsruhe/hzha03pyth6156/v99e/191.6/hzha03pyth6156_hgmm_191.6_110.0_3083.xsdst
Recid 83585: root://eospublic.cern.ch//eos/opendata/delphi/simulated-data/karlsruhe/hzha03pyth6156/v99e/191.6/hzha03pyth6156_hgmm_191.6_40.0_2943.xsdst
Recid 83595: root://eospublic.cern.ch//eos/opendata/delphi/simulated-data/ral/hzha03pyth6156/v99e/201.6/hzha03pyth6156_hgee_201.6_77.5_47762.xsdst
Recid 83597: root://eospublic.cern.ch//eos/opendata/delphi/simulated-data/ral/hzha03pyth6156/v99e/201.6/hzha03pyth6156_hgee_201.6_90.0_49012.xsdst
Recid 8

In [25]:
target = master_cache.get(str(recid), {})

In [18]:
file = "downloads/92941/EY1839.81.al"

for i in target.get("files", []):
    if i.get("path") == file:
        print(f"Found file {file} in recid {recid}:")
        print(json.dumps(i, indent=2))
        break
else:
    print(f"File {file} not found in recid {recid}.")

Found file downloads/92941/EY1839.81.al in recid 92941:
{
  "checksum": "adler32:00795fe4",
  "downloaded": true,
  "path": "downloads/92941/EY1839.81.al",
  "remote": "root://eospublic.cern.ch//eos/opendata/delphi/raw-data/y00/EY1839/EY1839.81.al",
  "size": 23116800
}


In [35]:
counter = 0
for file in problematic_files:
    found = False
    for recid, data in master_cache.items():
        for i in data.get("files", []):
            if i.get("path") == file:
                if i.get("downloaded"):
                    found = True
                else:
                    print(f"File {file} found in master_cache but not downloaded.")
                    counter += 1
    if not found:
        print(f"File {file} not found in master_cache.")
        counter += 1

print(f"Total problematic files found in master_cache but not downloaded: {counter}")

File downloads/92941/EY1839.81.al not found in master_cache.
File downloads/92941/EY1843.16.al not found in master_cache.
File downloads/92941/EY1788.61.al not found in master_cache.
File downloads/92941/EY1777.26.al not found in master_cache.
File downloads/92941/EY1823.55.al not found in master_cache.
File downloads/92941/EY1824.98.al not found in master_cache.
File downloads/92941/EY1831.42.al not found in master_cache.
File downloads/92941/EY1825.1.al not found in master_cache.
File downloads/92941/EY1832.89.al not found in master_cache.
File downloads/92941/EY1777.21.al not found in master_cache.
File downloads/92941/EY1775.39.al not found in master_cache.
File downloads/92941/EY1827.5.al not found in master_cache.
File downloads/92941/EY1796.27.al not found in master_cache.
File downloads/92941/EY1827.43.al not found in master_cache.
File downloads/92941/EY1843.9.al not found in master_cache.
File downloads/83967/run17640.xsdst not found in master_cache.
File downloads/92941/EY17